# 5.2 KDP 波长依赖性

**学习目标**
- 理解比差分相移KDP的物理机制
- 比较S、C、X波段的KDP差异
- 掌握KDP反演降雨率的公式

**参考文献**：Ryzhkov & Zrnic, Chapter 5, pp. 501-535

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def kdp_rainfall_relationship(R, wavelength_mm):
    """
    KDP与降雨率的经验关系
    基于 Ryzhkov & Zrnic (2019)
    """
    if wavelength_mm < 4:  # X-band
        return 0.062 * R**0.92
    elif wavelength_mm < 8:  # C-band
        return 0.028 * R**0.93
    else:  # S-band
        return 0.014 * R**0.95

def plot_kdp_wavelength():
    """可视化KDP的波长依赖性"""
    wavelengths = {'X-band (3.2cm)': 3.2, 'C-band (5.4cm)': 5.4, 'S-band (10.7cm)': 10.7}
    R_range = np.linspace(0.5, 100, 100)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = {'X-band (3.2cm)': 'red', 'C-band (5.4cm)': 'green', 'S-band (10.7cm)': 'blue'}
    
    # 子图1: KDP vs R
    ax1 = axes[0, 0]
    for band, wl in wavelengths.items():
        kdp = [kdp_rainfall_relationship(R, wl) for R in R_range]
        ax1.plot(R_range, kdp, color=colors[band], linewidth=2, label=band)
    ax1.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax1.set_ylabel('KDP (°/km)', fontsize=11)
    ax1.set_title('KDP vs 降雨率 (各波段)', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: KDP比值
    ax2 = axes[0, 1]
    kdp_x = [kdp_rainfall_relationship(R, 3.2) for R in R_range]
    kdp_c = [kdp_rainfall_relationship(R, 5.4) for R in R_range]
    kdp_s = [kdp_rainfall_relationship(R, 10.7) for R in R_range]
    ax2.plot(R_range, [kx/kc for kx, kc in zip(kdp_x, kdp_c)], 'r-', linewidth=2, label='X/C')
    ax2.plot(R_range, [kx/ks for kx, ks in zip(kdp_x, kdp_s)], 'b-', linewidth=2, label='X/S')
    ax2.plot(R_range, [kc/ks for kc, ks in zip(kdp_c, kdp_s)], 'g--', linewidth=2, label='C/S')
    ax2.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax2.set_ylabel('KDP 比值', fontsize=11)
    ax2.set_title('KDP 波段间比值', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 10)
    
    # 子图3: R(KDP) 反演
    ax3 = axes[1, 0]
    kdp_range = np.linspace(0.1, 5, 100)
    for band, wl in wavelengths.items():
        if wl < 4:
            R = (kdp_range / 0.062) ** (1/0.92)
        elif wl < 8:
            R = (kdp_range / 0.028) ** (1/0.93)
        else:
            R = (kdp_range / 0.014) ** (1/0.95)
        ax3.plot(kdp_range, R, color=colors[band], linewidth=2, label=band)
    ax3.set_xlabel('KDP (°/km)', fontsize=11)
    ax3.set_ylabel('反演降雨率 R (mm/h)', fontsize=11)
    ax3.set_title('从KDP反演降雨率', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 子图4: 典型KDP值表格
    ax4 = axes[1, 1]
    ax4.axis('off')
    table_data = []
    headers = ['R (mm/h)', 'KDP_X', 'KDP_C', 'KDP_S']
    for R in [1, 5, 10, 20, 50, 100]:
        table_data.append([R, kdp_rainfall_relationship(R,3.2), kdp_rainfall_relationship(R,5.4), kdp_rainfall_relationship(R,10.7)])
    table = ax4.table(cellText=table_data, colLabels=headers, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    ax4.set_title('各波段KDP典型值 (°/km)', fontsize=12, pad=20)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== KDP波段特性总结 ===")
    print("X波段 KDP 最大，但衰减最严重")
    print("S波段 KDP 最小，但衰减最小，适合远距离探测")
    print("KDP不受衰减影响，是降雨率估计的稳健参数")

In [ ]:
plot_kdp_wavelength()

## 练习题

1. **波段比较**：对于R=20mm/h的降雨，X波段和S波段的KDP分别为多少？比值是多少？

2. **反演**：X波段雷达测得KDP=2°/km，估计降雨率。

3. **KDP优势**：为什么KDP比ZDR更适合用于强降雨估计？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 5, Springer.